# Parking-occupancy recognition in snow with hidden road markings
## Research notebook v2: camera-held-out evaluation of winter adaptation

**Authors:** [Author 1], [Author 2] · [University] · v2, 30 October 2026

**Research question.** *How can computer-vision models be adapted to recognise parking-space occupancy accurately under heavy snowfall, when the painted road markings are hidden?*

### How to run
1. Open in **Google Colab**, then **Runtime → Change runtime type → T4 GPU**. The CPU also works, but feature extraction is much slower.
2. **Runtime → Run all.** The notebook downloads only the files it needs from the public SPKLv2 archive (≈0.7 GB of full frames and polygon files, using HTTP range requests).
   If the Google Drive download fails, put `SPKLv2.zip` in your Drive and set `LOCAL_ZIP` in §4.
3. Section §16 (the condition review) needs **both authors** to label frames by hand. Every other section runs automatically.
4. §19 saves `SnowParking_v2_results.zip`. It contains every table, figure and prediction file used in the paper.

### Status legend
| Mark | Meaning |
|---|---|
| ✅ v1 executed | Numbers from the 25 Oct 2026 pilot run. They are reproduced in §9 as a regression check. |
| ▶ v2 | Computed fresh by this notebook when you run it. **No v2 numbers are hard-coded anywhere.** |
| ✋ manual | Needs human labelling (§16). Results stay empty until the labels exist. |

## 1. Research overview and objectives

**Background.** Most camera-based parking systems either detect painted slot lines or classify fixed image regions. Snow defeats both: it hides the lines and changes how vehicles look. Our v1 pilot found that adding labelled winter images raised macro-F1 on held-out snowy cameras. However, it also tripled the rate at which occupied spaces were reported as free. A post-hoc audit of the v1 predictions (see the audit report) found three issues that this notebook fixes:

1. **Illumination confound.** 99 of the 100 v1 clear-weather training frames were daytime, but 51 % of the snowy test crops were night frames. The "winter" gain therefore mixed snow adaptation with night adaptation.
2. **Uncalibrated operating point.** The 0.5 threshold was never tuned on winter data. On the v1 predictions, comparing both models at the *same* false-free rate gave the adapted model higher free-space recall. This means the extra errors mostly came from where the threshold sat, not from worse ranking.
3. **A test camera with too few free spaces.** Camera 4 has only 70 free crops in total, so its per-camera F1 is noise-dominated.

**Objectives (v2).**
- **O1.** Compare four approaches under a strict *leave-one-snow-camera-out* protocol: source-only training, threshold calibration only, synthetic snow augmentation, and supervised winter adaptation.
- **O2.** Select every hyper-parameter and decision threshold on an **independent winter validation camera**, never on the test camera.
- **O3.** Evaluate at a **safety-constrained operating point**: false-free rate ≤ 5 % on validation.
- **O4.** Separate the snow effect from the illumination effect (day / night-RGB / infrared strata), and separate it from training-set size (size-matched control).
- **O5.** Estimate how many labelled winter frames are needed (learning curve).
- **O6.** Evaluate on a manually verified *heavy-snowfall + hidden-marking* subset, labelled independently by two annotators.

**Hypotheses** (tested automatically in §15):
- **H1** Real winter labels improve the ranking of free spaces. Test: AP(free) increases relative to the calibrated clear-only model, with the 95 % CI of the difference above 0.
- **H2** At the validation-selected safe threshold, the adapted model keeps the test false-free rate ≤ 7 % (5 % target + 2 pp tolerance) and achieves higher free-space recall than the calibrated clear-only model.
- **H3** The H1 gain holds in both the day and night strata, not only at night.
- **H4** Real winter labels outperform synthetic snow augmentation of clear images, measured by AP(free).

**Scope.** This study does not detect painted lines. It handles hidden markings with a **fixed-camera geometry prior**: the slot polygons are annotated once per camera, which is standard for fixed surveillance cameras. Inferring an unknown parking layout without markings is out of scope.

## 2. Problem definition

Each camera view $v$ has $S_v$ parking spaces with fixed polygons $P_{v,1},\dots,P_{v,S_v}$. For frame $I$ and space $s$, let $x_{s}=\mathrm{crop}(I,P_{v,s})$ and $y_s\in\{0{=}\text{free},1{=}\text{occupied}\}$.
A frozen encoder $\phi$ (ImageNet ResNet-18, 512-d) is followed by a logistic head:
$$\hat p_s = \sigma\big(w^\top \mathrm{std}(\phi(x_s)) + b\big),\qquad \hat y_s=\mathbb 1[\hat p_s\ge\tau].$$
The head is trained with class-balanced, L2-regularised cross-entropy (inverse regularisation strength $C$).

**Error that matters most.** A *false-free* error, reporting an occupied space as free, sends a driver to a space that does not exist. We therefore report
$$\mathrm{FFR}=\frac{FN_{occ}}{TP_{occ}+FN_{occ}},\qquad \mathrm{FreeRecall}=\frac{TN}{TN+FP},\qquad \mathrm{FreePrecision}=\frac{TN}{TN+FN_{occ}},$$
together with macro-F1, balanced accuracy, AP of the *free* class (threshold-independent, the minority class) and the Brier score.
The **safe operating point** $\tau_{safe}$ maximises free recall on the validation camera, subject to $\mathrm{FFR}_{val}\le 5\%$.

## 3. Dataset description

**SPKLv2** (Martynova et al., 2023, arXiv:2306.04288; code at github.com/Eighonet/parking-research) contains 1,203 annotated frames from fixed cameras. Each frame has per-space polygons with a binary occupancy label, plus image-level condition tags (`Snow`, `Night`, `Infrared`, `Sunny`, `Overcast`, `Rainy`, ...). The 440 `Snow` frames come from four camera views (`cam_1`…`cam_4`). The 763 non-snow frames carry no camera id, so §6 assigns them to views by matching their slot geometry.

The `Snow` tag marks snow-covered scenes. It does **not** separate falling snow from lying snow, and it does not record whether markings existed underneath. §16 adds that information by manual review. Image copyright belongs to the SPKL authors. This notebook downloads from the authors' public link and does not redistribute images.

## 4. Environment, configuration and seeds

In [ ]:
import importlib.util, subprocess, sys
_needed = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
           'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'ipywidgets': 'ipywidgets'}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

import base64, gzip, hashlib, io, json, os, platform, random, re, shutil, struct, time, urllib.parse, urllib.request, zlib
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, cohen_kappa_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- configuration (all experiment settings live here) ----------------
SEED = 2026
SMOKE_TEST = os.environ.get('SNOWPARK_SMOKE') == '1'      # only for code testing on mock data; never for results
LOCAL_ZIP = os.environ.get('SNOWPARK_LOCAL_ZIP')           # e.g. '/content/drive/MyDrive/SPKLv2.zip'
ZIP_URL_OVERRIDE = os.environ.get('SNOWPARK_ZIP_URL')      # direct URL that supports HTTP Range
GDRIVE_FILE_ID = '1281KUAFuChRZAODO-ZTWSE-W2ilzmdxm'       # public SPKLv2 archive (authors' link)

CROP_SIZE = 128                 # same as v1
C_GRID = [0.003, 0.03, 0.3]     # inverse L2 strength of the logistic head
SAFE_FFR = 0.05                 # false-free-rate budget used to choose tau_safe on validation
H2_TOLERANCE = 0.02             # allowed test overshoot of the budget for H2
AUG_SEEDS = [0, 1, 2]           # synthetic-snow augmentation seeds
SUBSAMPLE_SEEDS = [0, 1, 2, 3, 4]
LC_FRACTIONS = [0.1, 0.25, 0.5]  # learning curve: fraction of winter training frames (0 and 1 are B2 and W1)
N_BOOT = 2000                   # frame-cluster bootstrap replicates
RUN_MASK_ABLATION = True        # polygon-masked crops instead of bounding boxes
MIN_FREE_FOR_CAMERA_F1 = 100    # flag per-camera F1 when the free class is smaller than this

BASE = (Path('/content') if Path('/content').exists() else Path.cwd()) / 'snowpark_v2'
DATA, CROPS, RES, FIG = BASE / 'data', BASE / 'crops', BASE / 'results', BASE / 'results' / 'figures'
for p in (DATA, CROPS, RES, FIG):
    p.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'Pillow': Image.__version__, 'matplotlib': matplotlib.__version__,
            'device': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else platform.processor() or 'cpu'}
print(json.dumps(VERSIONS, indent=1))
print('Project folder:', BASE, '| SMOKE_TEST =', SMOKE_TEST)
if SMOKE_TEST:
    print('!!! SMOKE TEST MODE: mock data, random-init CNN. Numbers below are meaningless.')

In [ ]:
# Image-level condition tags published with SPKLv2 (1,203 frames), embedded so the notebook is self-contained.
META_B64 = 'H4sIAB2R42oC/7VdQY8etw39L3s2ihFJSWT/QNFLUzTHolgYjpu6SJ3AdlIURf973TTNOvWO5r0hddrDPPEbkY8SSVGz/3xov/rrd18//PqPD799++d3L9+9/urhxcPv3nz9lw8f/37x6tU3379/8+3bx1cv3z386cVDOzj0z/BfPnvx8IeXb97+4ycMLvPFw+9fv3v/3etXH9788PrxqzfvP3z77sPHhz8JalWCtEqQVQkaVYJmlaAoEtQAirQqijSpElRFkdarBFVRpHmVoCqKSJVjS5X55TnHpgT0rICZFeBJAQo4rh7ZH2lZAZoVkDW1jqyArKkNsJRJ9keyirasS1hW0ZZ1CYukgJ6le8+asWfp3rNm7Fm696wZB+AvI7uyjaypR9bhRtbUI+twI2vqkXW4CZh6Zk09s045s6aeWaecWVPPrFPOrKk9S3fPKtGzdPesEgOge2TpHtmVLbJki6yiI0k2rqjSgDJJO5Jm+RjxZwVoVoBnBWTN0iwrILkGPBUgFqaWrKmlZQVkFQ0khy2bHDbNMjqbHDYtqt40HVWCvEpQUfWmWasSJFWCrEpQlfltVgmqMn8HHLgfVT9Wxf5epcdexX4gl2yjSo+jykOGVgmq8pBRRZFR5SGjiiKzaoGcVeafVQvkrDL/rHLsWWX+n+WcDf7EwT0b6Xg2oPNspJPNnpuPrICZFZANaoMweTaLbpENbiNr8sjmOtk8Xo6WFSBZAZYV0LMCZlZAMut+SiavaS/tyP5Y1uRNswKyJm9Z2resyVtypXs6JwdMnq1WiGSdVLImz57Gi2RNnj2Nl2zFRrIn7ZKtxkj2pF00a8bsSbto1oxGFY7Fsr5nWatnj/3FslbPHvuLZa2ePfaXzlm9Z62e7RKQnrV6tktAetbq2S4B6VmrZzsAZGTNmO0AkJE1Y7YDQEbWjJPzvWwzgMys1WdRbU6qqjwyR5WgWSWoqDYnzrHDj6rflSpBVWzxXiWoii3uVYKq2BJVHhVVyo4qj4oiZSvlUHqQ8CL/06q7JFp1l0Sr7pJo1V0SrbpLolV3SZRru9GqayVada1Eq66VaNW1Eq26VqItGahptqil2a4YzRaqNHttRLOFKs1eG9FsoUrJDSDbJKTZupZm729otqqkmmU+V1XSbFVJs5dJNFtV0uxlEs1WlTR7mUSzVSXNXibRbJlIs5dJNFsm0uxlEs2WiTR7UUSzZSLNXhTRbJlIsxdFNFsm0uxFEc3WfTR7CUSzl0A0ewlEs5dANHsJRLOXQDTbSKOeNWO2jUUvulD+g+H2+6ftfoVJkseOkRXgWQFJ8li2N8GaZAVYVkDWjNneBMvmgSYAW7M9CZbNFS3bk2DZXNGyPQkmRfWdp2B+YTGVqh/TKkG9StCoEuRVgooKblZ1t8Oq7nZY1d0Oq7rb8SRnwf4BYBzAALFBbwAGWGOfcrsFBpj7U4a2wABzH1TMM4CYZwCaGqcX3b744fW7Vy/ff/gvzDDYwGATgwUE4w7A54EJPb3n+OX3b3/qHJwKYE7uIf34bJw/83Yt24F3dFv8xuLdfAKy/RoTlHHiOH+jADQSgEXCAMwAMICGIi4xQp0QyvkX6T6RKABGAUwHMAPAOIABNNUagAHm3gzAAHNv85St0vz8mVAWF8DiAmhGAIsLoBkBLC4TwAAWp85aRAFNPYXpv/nmo8BnQqFPsEpgO4EdBNYJbODYp8AXwF7o7EeMAZgOYCaA8WsM1ZH6SUPq/yQusNdR1fPtpQu4cfDBwScHp7Zs4d5lcLvd4JbKwal9cCQZ3FRn4+AcByb37pPjwKTaBoVEcxxwTo/OccA5tTvHAefU7pzrXSfLEpyug/O34HQdHGeD03X45ZcKJK6/H6XAB331EABjAKYDmAlgrueu7TQgVeD7TdoAtQAfi9AGTKcBZpLz6QgwHQEsKApgAAsKoBYBLCiAWhQwkwJzV4C9Csxdz3MkVUdKMXpewfik0WctoWEwxWCGwQYGmxgMqoXpeUVHO6arLhgM01XvGAzTVffF9OL82YKDixqcztW4xc8t3tKPxbPrNEtdAUwHMAPAOICJa0wsdp5YLOOxcPxYED2ooEuDil6pcNGuCxJ2HJzExsEHB58cnIpX7bpIZdz/57AmHFw5eOfgnK65q9LG3fMzLicz7p6KcRcVjOuZNq7dlzme/hHOKVI5jhunSOMIbJwijSOwcYo0jmLGKbKf7xrWZfHMFs/64tlcPDvf0e26Gm7jWAxfTPPsbPbHZ+fxh10X320uXmkutDsXr7Q46rQJqGk6gLk+NDDg6NSAo1NzAzAdwAD2AI5SDZh6AAEHcI5qwDmqAeeoBpyjGnCOasA5KrXU9Ws99QPBCIBRAHPuOv0Yi2e+eAZljL01DCYY7Lps0IFPs3agStTbdcmkA/YBqkVdGoBRAAOoB6gWdQHUA1SLOkBNBdQDFJS6AuoBCkpdAfUo/m3LzsV33RoHFw5uHJxb9GxycCq+69c7Ye8HJ5HTNXfo2rlD184duvbrja87AOEUxh2VdWBT5Q6EOldL6UH53uCqE+OgXmZch0WDq04MrtwwuPrB4Poex/VOOK4DunHt5OM6nBvXzB9cEY8DXwd28zhPr+Y1UWY7FsNl8UwXz/ri2XmMOKUtnp0ny0Bz69TFNBfnjVMX2r0m4bTzqHcu0n8gB52LlHteE3suUuc5V6I7IBt4fSCFnkAKPYHUdwKp7wRS3wmkvsgrA6nvBFLfCaS+E0h9J5D6zphwE94MvLmP2vf8Wm9+HPhvH0Jgr0+z/OgAZgAYBzDn65K3tngmi2fXXHGg2djbBDDXruSLVwU6jR3oNHag09iBTmMHOo0d6DR2oNPYzxdnB9qKXaHyiKtisH6Vv/4Sfn37ypVbFLis2Lms2Lms2Lms2Lms2Lms2Lms2KH+B+cSY+cSY+cSY+cSY+cSY+faix1qDHGurdi5tmLn2oqdayt2rq3YB8e/wamba1r2ySmS63F2rsfZuR5n53qcHepVc65g41zBxv26hupuAGYAmAlgrkvMDp1CeByAJAEwgIaiA5hrDQXQKBxAo3AAjcIBNAoH0CgcQKNwANMCGoajIXIUwADqARqPAzhSCqDxGGBhAEdKIeeBeIgunvXFs/MyUCzaaGPRNhqL5slYdDoGYDU7FsPb4tlCPbaYpi3UY4tp2nlaCKwl0RfTXHSyBHA6F8CpSwBnEQEcRgRwXhBAFT2AUnQAtegAitEBVKMDKEcHsHZgHxaI2TCYYjDDYAODLZxgUoEkcBknuFgpXDi4cvDOwQcHdw7O6ToaB+cUGVTQ/url3x7b43FVv/jy7bd/f4I3Es6KP+gB9BsJO0DZAcYO6OyAwQ6Y7ABnBwQ5gLUbTSX+F64rg5+P0Rtj7MaYfmPMuDFm3hjjN8YEP4Z1XDlu/Ea7MeYGb+QGb+QGb+QGb+QGb+QGb+QGb4RdZtjFW9llRtllRlkeKz0HdgNSdgNSdgNSdgNSdgNSlhmsklgdsSqCPeiTbOenoedp0QmefTdnzees+Zw1X5BzDtaxg3XsYB07WMcOlrPBkjZYZgTLjGCZESQzSCssjtKfJZKwKYqwKcriq3CnG+PiK3GLMXZjTL8xZtwYM2+M8Rtj+EB00VjxPGPaceM32o0xN3hzI4GRGwmM3Ehg5EYCIzcSGLmRwHzyf89BpxaSNMIuM8J8IPn/x7KLpig7wNgBPTGdwf7YZAc4O4Dli5J8MRLfSfwg8Tf88IYb8qs3SVzl127ll27lV27lF242LzRyVWDTSDaLZJNINodkU0hWneSKbmTcaCS1jXx/0rqkcUnbkqYlLcumQWwWxCZBXA4kj+2aayf/R+BzSa1MkpRJ0jJJViapl0kaZZJmmSQvkxRVknoZx3sZx3sZx3sZx3sZx3sZx3sZx/tpoAlL8LSEyEoYR1pCS0uQtISelqBpCZaWMNIS0pwcaU6ONCdnmpMzzcmZ5uRMM2qmGTUL/sHT50LHDqFzh1DfITQ2CCUa3QihbYdQ2SFUdwi1HUI3eJTosUNo2yFUdgjVHUJth9At1h87hM4dQn2H0A3rqdgOj7IdHmU7PMp2eJTt8Cjb4VGW+LewnwvLxtRinpaQjamlH2kJLS1B0hI0LcHSEnpawkhLSHMyXXuQdO1B0rUHSdceJF17kHTlQNKVg+f/TdVtYTvikbEjHhk74pGxIx6ZO+KRuSMemTvikbkjHpkb4pEd4Yht8Cfb4E62wZtsgzP1Db7UN7hS3+BJfYMj9Q1+1Df4Ud/gR32DH/UNftQ3+NHY4Eej1QUiz36C564sLZRlhbIKA7dRmPU+G67dleWFsqJO1rMx2V1Zhbyfhbyfhbyfhbyfhbyfhbyfhbyfhbyfhbz3Qt57Ie+9kPdeyHsv5P2O4yPfEBf5hrjIN8RFviEuig1xUWzIL2JDfhEb8ovYkF9EuR/p4/o+3s+wBsKopmv99AskV0D4DQQFKgo0FNhR4ECBEwQ2Uu9C4pXEG4nvJH6Q+EnincQHhyfVL6RbCUkHYd+HpIOQdBDQkbhrUrq4JnWCdxIfHJ5Uox6k+EbihcQriTcS30k8SQcl6aCO0VLJ1YD0DiNXAyNXAyNXAyNpbOx8yc3ByM3ByM3ByM3BSDqQ0+0kHTpJh07SoZN06CQdOqsfkg6dpEMn6dBJOpCvP0g6DJIOA4yvB7k5DHJzGOTmMMjNYZCbwyBjhUHGCiQrJxkrTDJWmGSsMEk6TJIOk6TDJN1qsvonV4VJrgqkeCdXBSdXBSc3CSc3CSc3CSc3CSfp4CQdnLUXSQcWTtIhSDoESYcg6RAkHYKkQ5B0CJIOQdIhAtl0jfu+sHHfF7bHdmBv0RqIExCnIM5AHLVX2GMbJH6SeCfxweGFMzP3jTYjv9Bl5MeZjPw4j5Ef5zHy4zxGVp2MrDoZWXUysupkXNXJyKqTkVUnI6tORladjKw6GVl1MrLqZGTVya6qTp/jSToYRwcj6WAkHYykg5F0MHCT4KpORladjKw6GVl1MrLqZI/kdDsZanQy1ujkptXJVa2TtO+sfkg6kHCSPCR3QOr8698Iz+tzZfIAAA=='
FRAME_TAGS = json.loads(gzip.decompress(base64.b64decode(META_B64)))
# The 100 clear-weather frames used in the v1 pilot (needed only for the §9 replication).
PILOT_CLEAR_FRAMES = ["164.jpg", "167.jpg", "169.jpg", "172.jpg", "173.jpg", "182.jpg", "187.jpg", "191.jpg", "194.jpg", "196.jpg", "199.jpg", "200.jpg", "203.jpg", "206.jpg", "208.jpg", "211.jpg", "212.jpg", "215.jpg", "220.jpg", "223.jpg", "226.jpg", "227.jpg", "230.jpg", "232.jpg", "235.jpg", "236.jpg", "239.jpg", "338.jpg", "340.jpg", "343.jpg", "344.jpg", "347.jpg", "349.jpg", "352.jpg", "355.jpg", "356.jpg", "40.jpg", "41.jpg", "47.jpg", "478.jpg", "479.jpg", "482.jpg", "484.jpg", "487.jpg", "49.jpg", "490.jpg", "491.jpg", "494.jpg", "496.jpg", "499.jpg", "502.jpg", "503.jpg", "511.jpg", "512.jpg", "52.jpg", "53.jpg", "56.jpg", "59.jpg", "61.jpg", "64.jpg", "65.jpg", "68.jpg", "70.jpg", "73.jpg", "76.jpg", "77.jpg", "776.jpg", "778.jpg", "784.jpg", "787.jpg", "788.jpg", "791.jpg", "793.jpg", "796.jpg", "797.jpg", "80.jpg", "800.jpg", "814.jpg", "815.jpg", "818.jpg", "820.jpg", "823.jpg", "824.jpg", "827.jpg", "830.jpg", "832.jpg", "835.jpg", "85.jpg", "88.jpg", "89.jpg", "953.jpg", "956.jpg", "958.jpg", "961.jpg", "964.jpg", "965.jpg", "968.jpg", "970.jpg", "973.jpg", "976.jpg"]
if SMOKE_TEST and os.environ.get('SNOWPARK_SMOKE_TAGS'):
    FRAME_TAGS = json.loads(Path(os.environ['SNOWPARK_SMOKE_TAGS']).read_text())
    PILOT_CLEAR_FRAMES = [n for n, t in FRAME_TAGS.items() if 'Snow' not in t][:20]
print(len(FRAME_TAGS), 'frames with tags;', sum('Snow' in t for t in FRAME_TAGS.values()), 'tagged Snow')

## 5. Data loading
The archive is ≈3.2 GB, so the notebook does not download all of it. It reads the ZIP central directory, then fetches only the full frames and polygon files, using HTTP range requests (or a local file). Every member is CRC-checked.

In [ ]:
class ZipSource:
    """Random access to a ZIP archive stored in a local file or behind a URL that supports HTTP Range."""
    def __init__(self, url: str = None, path: str = None):
        assert (url is None) != (path is None)
        self.url, self.path = url, path
        if path:
            self.size = os.path.getsize(path)
        else:
            self.size = int(self._get(0, 0, want_total=True))

    def _get(self, start: int, end: int, want_total: bool = False):
        for attempt in range(5):
            try:
                req = urllib.request.Request(self.url, headers={'Range': f'bytes={start}-{end}'})
                with urllib.request.urlopen(req, timeout=180) as r:
                    if r.status != 206:
                        raise RuntimeError('Server ignored the Range header; refusing to download the whole archive.')
                    total = int(r.headers['Content-Range'].split('/')[-1])
                    if want_total:
                        return total
                    data = r.read()
                if len(data) != end - start + 1:
                    raise IOError('incomplete range')
                return data
            except Exception as exc:  # network hiccups are common on Colab
                if attempt == 4:
                    raise
                print('  retrying range request:', exc); time.sleep(2 * (attempt + 1))

    def read(self, start: int, length: int) -> bytes:
        if self.path:
            with open(self.path, 'rb') as f:
                f.seek(start); return f.read(length)
        return self._get(start, start + length - 1)

def resolve_gdrive_url(file_id: str) -> str:
    """Public Google Drive files larger than ~100 MB show a confirmation form; parse it (no credentials used)."""
    first = f'https://drive.google.com/uc?export=download&id={file_id}'
    with urllib.request.urlopen(first, timeout=60) as r:
        head = r.read(200_000)
        final = r.geturl()
    if b'<form' not in head:  # served directly
        return final
    params = dict(re.findall(r'name="([^"]+)" value="([^"]*)"', head.decode('utf-8', 'ignore')))
    if params.get('id') != file_id:
        raise RuntimeError('Unexpected Google Drive page. Download SPKLv2.zip manually and set LOCAL_ZIP.')
    return 'https://drive.usercontent.google.com/download?' + urllib.parse.urlencode(params)

def zip_directory(src: ZipSource) -> dict:
    """Parse the central directory (ZIP and ZIP64) -> {name: (local_header_offset, compressed_size, size, method, crc)}."""
    tail_len = min(src.size, 1 << 17)
    tail = src.read(src.size - tail_len, tail_len)
    eocd = tail.rfind(b'PK\x05\x06')
    if eocd < 0:
        raise RuntimeError('Not a ZIP archive')
    cd_size, cd_offset = struct.unpack_from('<II', tail, eocd + 12)
    loc = tail.rfind(b'PK\x06\x07')
    if loc >= 0:  # ZIP64 end-of-central-directory locator present
        z64 = struct.unpack_from('<Q', tail, loc + 8)[0]
        rec = src.read(z64, 56)
        cd_size, cd_offset = struct.unpack_from('<QQ', rec, 40)
    cd = src.read(cd_offset, cd_size)
    entries, i = {}, 0
    while i < len(cd) and cd[i:i + 4] == b'PK\x01\x02':
        method, = struct.unpack_from('<H', cd, i + 10)
        crc, csize, usize = struct.unpack_from('<III', cd, i + 16)
        nlen, elen, clen = struct.unpack_from('<HHH', cd, i + 28)
        offset, = struct.unpack_from('<I', cd, i + 42)
        name = cd[i + 46:i + 46 + nlen].decode('utf-8', 'replace')
        extra = cd[i + 46 + nlen:i + 46 + nlen + elen]
        j = 0
        while j + 4 <= len(extra):  # ZIP64 extended information
            tag, size = struct.unpack_from('<HH', extra, j)
            if tag == 1:
                vals, k = [], j + 4
                for flag in (usize == 0xFFFFFFFF, csize == 0xFFFFFFFF, offset == 0xFFFFFFFF):
                    if flag:
                        vals.append(struct.unpack_from('<Q', extra, k)[0]); k += 8
                it = iter(vals)
                if usize == 0xFFFFFFFF: usize = next(it)
                if csize == 0xFFFFFFFF: csize = next(it)
                if offset == 0xFFFFFFFF: offset = next(it)
            j += 4 + size
        entries[name] = (offset, csize, usize, method, crc)
        i += 46 + nlen + elen + clen
    return entries

def fetch_members(src: ZipSource, members: dict, max_gap: int = 2 << 20, max_span: int = 48 << 20) -> None:
    """members: {zip_name: destination Path}. Adjacent members are fetched in merged spans."""
    todo = sorted(((DIRECTORY[n][0], n, dst) for n, dst in members.items() if not dst.exists()))
    spans, cur = [], []
    for off, name, dst in todo:
        end = off + DIRECTORY[name][1] + 1024
        if cur and (off - cur_end > max_gap or end - cur[0][0] > max_span):
            spans.append(cur); cur = []
        if not cur:
            cur_end = end
        cur.append((off, name, dst)); cur_end = max(cur_end, end)
    if cur:
        spans.append(cur)
    for k, span in enumerate(spans):
        lo = span[0][0]; hi = max(o + DIRECTORY[n][1] + 1024 for o, n, _ in span)
        blob = src.read(lo, min(hi, src.size) - lo)
        for off, name, dst in span:
            _, csize, usize, method, crc = DIRECTORY[name]
            d = blob[off - lo:]
            assert d[:4] == b'PK\x03\x04', name
            nlen, elen = struct.unpack_from('<HH', d, 26)
            raw = d[30 + nlen + elen:30 + nlen + elen + csize]
            data = zlib.decompress(raw, -15) if method == 8 else raw
            if len(data) != usize or (zlib.crc32(data) & 0xFFFFFFFF) != crc:
                raise IOError(f'Corrupted member {name}')
            dst.parent.mkdir(parents=True, exist_ok=True); dst.write_bytes(data)
        print(f'  span {k + 1}/{len(spans)} ({len(span)} files)', flush=True)

if LOCAL_ZIP:
    SOURCE = ZipSource(path=LOCAL_ZIP)
else:
    SOURCE = ZipSource(url=ZIP_URL_OVERRIDE or resolve_gdrive_url(GDRIVE_FILE_ID))
DIRECTORY = zip_directory(SOURCE)
print(f'Archive: {SOURCE.size / 1e9:.2f} GB, {len(DIRECTORY):,} members')

def locate(basename: str, folder_hint: str) -> str:
    """Pick the full-frame member for a frame (prefer the main SPKLv2/<split>/<folder_hint>/ folder)."""
    cands = [n for n in BY_BASENAME.get(basename, []) if f'/{folder_hint}/' in n or folder_hint == 'Winter']
    cands = [n for n in cands if 'patch' not in n.lower()]
    if not cands:
        return None
    return sorted(cands, key=lambda n: (f'/{folder_hint}/' not in n, len(n)))[0]

BY_BASENAME = defaultdict(list)
for _n in DIRECTORY:
    BY_BASENAME[_n.rsplit('/', 1)[-1]].append(_n)

wanted, missing = {}, []
for frame in sorted(FRAME_TAGS):
    stem = frame.rsplit('.', 1)[0]
    img = locate(frame, 'images') or locate(frame, 'Winter')
    mk = locate(stem + '.json', 'int_markup')
    if img is None or mk is None:
        missing.append(frame); continue
    wanted[img] = DATA / 'frames' / frame
    wanted[mk] = DATA / 'markup' / (stem + '.json')
print(f'Fetching {len(wanted) // 2} frames with polygons; {len(missing)} tagged frames not found in archive')
fetch_members(SOURCE, wanted)
print('Data ready.')

## 6. Data preprocessing: frame table, camera-view assignment and crops
* **Condition strata.** `illum` takes the value *infrared*, *night* (RGB at night) or *day*, from the image-level tags.
* **View assignment.** Snow frames take their camera from the file name (`cam_k_*`). Non-snow frames are assigned to `cam_k` when they have the same number of slots and the same image size as `cam_k`, and their slot centroids lie within 2 % of the image width of `cam_k`'s (symmetric nearest-centroid distance). Unmatched frames are grouped into `other_*` views by their exact geometry. This replaces v1's exact-hash match, which silently pooled every unmatched frame into one pseudo-camera.
* **Deduplication.** Byte-identical frames are removed, as in v1.
* **Crops.** v1 used the polygon's bounding box, resized to 128². The ablation in §13 also builds a polygon-masked variant, where pixels outside the polygon are set to the ImageNet mean colour.

In [ ]:
def load_markup(frame: str) -> list:
    j = json.loads((DATA / 'markup' / (frame.rsplit('.', 1)[0] + '.json')).read_text())
    return [(np.asarray(l['coordinates'], dtype=float), int(l['label'])) for l in j['lots']]

def centroids(lots) -> np.ndarray:
    return np.array([poly.mean(0) for poly, _ in lots])

def geometry_distance(a: np.ndarray, b: np.ndarray) -> float:
    if len(a) != len(b):
        return np.inf
    d = np.linalg.norm(a[:, None] - b[None], axis=2)
    return 0.5 * (d.min(1).mean() + d.min(0).mean())

records, seen_hash = [], {}
for f in sorted((DATA / 'frames').glob('*.jpg')):
    tags = FRAME_TAGS.get(f.name, [])
    with Image.open(f) as im:
        h = hashlib.sha256(im.tobytes()).hexdigest(); size = im.size
    if h in seen_hash:
        continue
    seen_hash[h] = f.name
    lots = load_markup(f.name)
    m = re.match(r'cam_(\d+)_', f.name)
    records.append(dict(frame=f.name, snow='Snow' in tags,
                        illum='infrared' if 'Infrared' in tags else ('night' if 'Night' in tags else 'day'),
                        tags='|'.join(tags), width=size[0], height=size[1], n_spots=len(lots),
                        view=f'cam{m.group(1)}' if m else None, sha256=h))
frames = pd.DataFrame(records)

# Reference geometries of the snow cameras (all distinct geometries observed per camera)
REF = defaultdict(list)
for r in frames[frames.view.notna()].itertuples():
    c = centroids(load_markup(r.frame))
    if not any(geometry_distance(c, x) < 1e-6 for x, _ in REF[r.view]):
        REF[r.view].append((c, (r.width, r.height)))
other_sigs = {}
for i, r in frames[frames.view.isna()].iterrows():
    c = centroids(load_markup(r.frame)); best, best_d = None, np.inf
    for view, geoms in REF.items():
        for g, wh in geoms:
            if wh == (r.width, r.height):
                d = geometry_distance(c, g)
                if d < best_d:
                    best, best_d = view, d
    if best is not None and best_d <= 0.02 * r.width:
        frames.at[i, 'view'] = best
    else:
        sig = hashlib.sha1(np.round(c / 5).astype(int).tobytes() + str((r.width, r.height)).encode()).hexdigest()[:6]
        frames.at[i, 'view'] = other_sigs.setdefault(sig, f'other_{len(other_sigs) + 1:02d}')
print('Distinct geometries per snow camera:', {k: len(v) for k, v in REF.items()})
print(f'{len(frames)} unique frames (removed {len(FRAME_TAGS) - len(missing) - len(frames)} exact duplicates)')

In [ ]:
def build_crops(frames: pd.DataFrame) -> pd.DataFrame:
    rows, mean_rgb = [], (124, 116, 104)
    (CROPS / 'bbox').mkdir(exist_ok=True); (CROPS / 'mask').mkdir(exist_ok=True)
    for r in frames.itertuples():
        with Image.open(DATA / 'frames' / r.frame) as raw:
            im = raw.convert('RGB')
        for k, (poly, label) in enumerate(load_markup(r.frame)):
            assert label in (0, 1)
            x0, y0 = np.floor(poly.min(0)).astype(int); x1, y1 = np.ceil(poly.max(0)).astype(int)
            x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(im.width, x1 + 1), min(im.height, y1 + 1)
            if x1 <= x0 or y1 <= y0:
                raise ValueError(f'Invalid polygon in {r.frame}, spot {k}')
            stem = f"{r.frame.rsplit('.', 1)[0]}_s{k:02d}.jpg"
            pb, pm = CROPS / 'bbox' / stem, CROPS / 'mask' / stem
            if not pb.exists():
                crop = im.crop((x0, y0, x1, y1))
                crop.resize((CROP_SIZE, CROP_SIZE)).save(pb, quality=93)
                mask = Image.new('L', crop.size, 0)
                ImageDraw.Draw(mask).polygon([tuple(p) for p in (poly - [x0, y0])], fill=255)
                Image.composite(crop, Image.new('RGB', crop.size, mean_rgb), mask).resize((CROP_SIZE, CROP_SIZE)).save(pm, quality=93)
            rows.append(dict(frame=r.frame, spot=k, label=label, view=r.view, snow=r.snow, illum=r.illum,
                             bbox=str(pb.relative_to(BASE)), mask=str(pm.relative_to(BASE)),
                             polygon=json.dumps(poly.round(1).tolist()), box_area=(x1 - x0) * (y1 - y0)))
    return pd.DataFrame(rows)

crops = build_crops(frames)
crops.to_csv(RES / 'crops_manifest.csv', index=False)
y_all = crops.label.to_numpy()
print(f'{len(crops):,} crops from {crops.frame.nunique()} frames')

## 7. Exploratory data analysis

In [ ]:
eda = (crops.groupby(['view', 'snow', 'illum'])
       .agg(frames=('frame', 'nunique'), crops=('label', 'size'), free=('label', lambda s: int((s == 0).sum())),
            occupied_share=('label', 'mean')).reset_index())
eda['occupied_share'] = eda.occupied_share.round(3)
display(eda)
eda.to_csv(RES / 'eda_view_condition.csv', index=False)
snow_free = crops[crops.snow].groupby('view').label.apply(lambda s: int((s == 0).sum()))
print('Free crops per snow camera:', snow_free.to_dict())
SMALL_FREE_VIEWS = [v for v, n in snow_free.items() if n < MIN_FREE_FOR_CAMERA_F1]
print('Cameras whose free class is too small for stable per-camera F1:', SMALL_FREE_VIEWS)

# Contact sheet: one snow frame per camera with annotated polygons (labels from annotation, not model output)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, view in zip(axes, ['cam1', 'cam2', 'cam3', 'cam4']):
    sub = frames[(frames.view == view) & frames.snow].sort_values('frame')
    ax.axis('off')
    if sub.empty:
        continue
    fr = sub.iloc[len(sub) // 2].frame
    with Image.open(DATA / 'frames' / fr) as raw:
        im = raw.convert('RGB')
    d = ImageDraw.Draw(im)
    for poly, lab in load_markup(fr):
        d.line([tuple(p) for p in poly] + [tuple(poly[0])], fill=(230, 60, 60) if lab else (40, 200, 90), width=4)
    ax.imshow(im); ax.set_title(f'{view} · {fr}', fontsize=9)
plt.suptitle('Annotated slot polygons on snow frames (red = occupied, green = free)', fontsize=10)
plt.tight_layout(); plt.savefig(FIG / 'fig1_dataset.png', dpi=200); plt.show()

## 8. Feature extraction (frozen ImageNet ResNet-18) and synthetic-snow baseline
The encoder is frozen, as in v1, so every experiment shares one cached feature matrix and the comparisons isolate the *training data*, not the optimisation.
**Synthetic snow** (baseline S1) is a heuristic, seeded generator applied to clear-weather crops. It has three components: low-frequency ground snow cover, which also covers markings; atmospheric haze; and falling flakes with motion streaks. It needs no winter labels.

In [ ]:
_NORM = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                        torchvision.transforms.Normalize([.485, .456, .406], [.229, .224, .225])])

def synthetic_snow(img: Image.Image, rng: np.random.Generator) -> Image.Image:
    a = np.asarray(img, dtype=np.float32) / 255.0; h, w, _ = a.shape
    from PIL import ImageFilter
    noise = Image.fromarray((rng.random((5, 5)) * 255).astype(np.uint8)).resize((w, h), Image.BICUBIC)
    low = np.asarray(noise.filter(ImageFilter.GaussianBlur(w / 12)), np.float32) / 255
    low = (low - low.min()) / (np.ptp(low) + 1e-6)
    cover = rng.uniform(0.25, 0.75)                                   # fraction of ground covered
    mask = np.clip((cover - low) * 2.5 + 0.5, 0, 1)[..., None] * rng.uniform(0.45, 0.85)
    a = a * (1 - mask) + rng.uniform(0.82, 0.97) * mask               # lying snow (hides lines)
    haze = rng.uniform(0.05, 0.35); a = a * (1 - haze) + haze * rng.uniform(0.7, 0.85)
    out = Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8)); d = ImageDraw.Draw(out)
    angle = rng.uniform(-0.6, 0.6)
    for _ in range(int(rng.integers(15, 90))):                       # falling flakes
        x, y, L = rng.uniform(0, w), rng.uniform(0, h), rng.uniform(1, 6)
        d.line([(x, y), (x + L * np.sin(angle), y + L * np.cos(angle))], fill=(245, 245, 245), width=int(rng.integers(1, 3)))
    return out

class CropSet(torch.utils.data.Dataset):
    def __init__(self, paths, aug_seed=None):
        self.paths, self.aug_seed = list(paths), aug_seed
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(BASE / self.paths[i]) as im:
            im = im.convert('RGB')
        if self.aug_seed is not None:
            im = synthetic_snow(im, np.random.default_rng([self.aug_seed, i]))
        return _NORM(im)

_ENCODER = None
def encoder():
    global _ENCODER
    if _ENCODER is None:
        weights = None if SMOKE_TEST else torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        m = torchvision.models.resnet18(weights=weights); m.fc = torch.nn.Identity()
        _ENCODER = m.to(DEVICE).eval()
    return _ENCODER

def embed(paths, tag: str, aug_seed=None) -> np.ndarray:
    key = hashlib.sha256(('\n'.join(paths) + f'|{aug_seed}|{SMOKE_TEST}').encode()).hexdigest()[:16]
    cache = BASE / 'features' / f'{tag}_{key}.npy'; cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        return np.load(cache)
    loader = torch.utils.data.DataLoader(CropSet(paths, aug_seed), batch_size=128, shuffle=False,
                                         num_workers=2 if DEVICE == 'cuda' else 0)
    out, t0 = [], time.time()
    with torch.inference_mode():
        for b in loader:
            out.append(encoder()(b.to(DEVICE)).float().cpu().numpy())
    x = np.concatenate(out).astype(np.float32); np.save(cache, x)
    print(f'  {tag}: {len(paths):,} crops embedded in {time.time() - t0:.0f}s')
    return x

TIMING = {}
t0 = time.time(); X = {'bbox': embed(crops.bbox.tolist(), 'bbox')}; TIMING['embed_bbox_s'] = time.time() - t0
TIMING['embed_ms_per_crop'] = 1000 * TIMING['embed_bbox_s'] / len(crops)
clear_idx_all = np.where(~crops.snow.to_numpy())[0]
X_AUG = {}
for s in AUG_SEEDS:  # augmented copies exist only for clear crops (row-aligned with clear_idx_all)
    X_AUG[s] = embed(crops.bbox.to_numpy()[clear_idx_all].tolist(), f'synsnow{s}', aug_seed=s)
if RUN_MASK_ABLATION:
    X['mask'] = embed(crops['mask'].tolist(), 'mask')

# Visual check of the synthetic generator
ex = crops.iloc[clear_idx_all[:6]]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for j, (_, r) in enumerate(ex.iterrows()):
    with Image.open(BASE / r.bbox) as im:
        im = im.convert('RGB')
    axes[0, j].imshow(im); axes[1, j].imshow(synthetic_snow(im, np.random.default_rng([0, j])))
    for a in axes[:, j]: a.axis('off')
axes[0, 0].set_title('clear crop', loc='left', fontsize=9); axes[1, 0].set_title('synthetic snow (S1)', loc='left', fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'fig_synthetic_snow.png', dpi=200); plt.show()

## 9. Evaluation utilities and v1 pilot replication (regression check)
The v1 pilot recorded these numbers (✅ v1 executed, 25 Oct 2026): clear baseline macro-F1 **0.6070**, false-free rate **5.39 %**; winter-adapted macro-F1 **0.6873**, false-free rate **17.52 %**. The cell below re-implements the v1 protocol on the v2 crops and prints both, so any difference is visible. (It can differ slightly because of library versions and floating-point precision.)

In [ ]:
def fast_metrics(y: np.ndarray, p: np.ndarray, dec: np.ndarray, with_ap: bool = True) -> dict:
    """All metrics of §2. dec is the occupied decision (1/0), or the fraction of seeds voting occupied (majority >= 0.5).
    macro-F1 counts an absent class as F1 = 0 (same as scikit-learn with zero_division=0)."""
    y = np.asarray(y); p = np.asarray(p, dtype=float); occ, pred = y == 1, np.asarray(dec) >= 0.5
    tp, fn = np.sum(pred & occ), np.sum(~pred & occ); fp, tn = np.sum(pred & ~occ), np.sum(~pred & ~occ)
    div = lambda a, b: a / b if b else np.nan
    f1_occ, f1_free = div(2 * tp, 2 * tp + fp + fn), div(2 * tn, 2 * tn + fn + fp)
    return dict(n=len(y), n_free=int((~occ).sum()),
                macro_f1=np.nansum([f1_occ, f1_free]) / 2,
                balanced_acc=0.5 * (div(tp, tp + fn) + div(tn, tn + fp)),
                false_free_rate=div(fn, tp + fn), free_recall=div(tn, tn + fp), free_precision=div(tn, tn + fn),
                ap_free=average_precision_score(1 - y, 1 - p) if with_ap and 0 < occ.sum() < len(y) else np.nan,
                brier=float(np.mean((p - y) ** 2)), tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn))

def metrics(y, p, tau: float = 0.5, with_ap: bool = True) -> dict:
    return fast_metrics(y, p, (np.asarray(p) >= tau).astype(float), with_ap)

def choose_thresholds(y_val, p_val) -> dict:
    """Exhaustive search over all distinct validation scores (vectorised).
    tau_f1: maximises macro-F1. tau_safe: maximises free recall subject to FFR <= SAFE_FFR. Validation data only."""
    y = np.asarray(y_val); p = np.asarray(p_val, dtype=float)
    order = np.argsort(p, kind='stable'); ps, ys = p[order], y[order]
    cuts = np.concatenate([[0], np.where(np.diff(ps) > 0)[0] + 1, [len(ps)]])   # predict free for the first `cut` rows
    taus = np.append(ps, np.nextafter(ps[-1], np.inf))[cuts]
    fn = np.concatenate([[0], np.cumsum(ys)])[cuts]; tn = np.concatenate([[0], np.cumsum(1 - ys)])[cuts]
    n_occ, n_free = ys.sum(), len(ys) - ys.sum(); tp, fp = n_occ - fn, n_free - tn
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = (np.nan_to_num(2 * tp / (2 * tp + fp + fn)) + np.nan_to_num(2 * tn / (2 * tn + fn + fp))) / 2
        ffr = np.where(n_occ > 0, fn / max(n_occ, 1), 0.0); free_rec = tn / max(n_free, 1)
    safe = np.where(ffr <= SAFE_FFR, free_rec, -1)
    return {'default': 0.5, 'f1': float(taus[int(np.argmax(f1))]), 'safe': float(taus[int(np.argmax(safe))])}

def logistic_head(C: float):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=3000, random_state=SEED))

def pilot_replication() -> pd.DataFrame:
    """v1 protocol: 100 pilot clear frames; cam1 clear -> train, other clear -> val; snow cam1/2 -> winter train, cam3/4 -> test."""
    c = crops.copy(); x = X['bbox']
    pilot_view = {}
    for fr in frames.frame:  # v1 used an exact hash of the raw polygon coordinates
        lots = json.loads((DATA / 'markup' / (fr.rsplit('.', 1)[0] + '.json')).read_text())['lots']
        pilot_view[fr] = hashlib.sha256(json.dumps([l['coordinates'] for l in lots]).encode()).hexdigest()
    ref = {}
    for k in range(1, 5):
        firsts = sorted(f for f in frames.frame if f.startswith(f'cam_{k}_'))
        if firsts:
            ref[pilot_view[firsts[0]]] = f'cam{k}'
    cam = c.frame.map(lambda f: ref.get(pilot_view[f], 'cam5'))
    pilot_frames = set(PILOT_CLEAR_FRAMES)
    split = np.where(c.snow, np.where(cam.isin(['cam1', 'cam2']), 'winter_train', 'snow_test'),
                     np.where(~c.frame.isin(pilot_frames), 'unused', np.where(cam == 'cam1', 'clear_train', 'clear_val')))
    tr, va, wi, te = (split == s for s in ('clear_train', 'clear_val', 'winter_train', 'snow_test'))
    if tr.sum() == 0 or va.sum() == 0 or te.sum() == 0:
        print('Replication not possible with the available frames.'); return pd.DataFrame()
    val = []
    for C in [0.01, 0.1, 1.0]:
        m = logistic_head(C).fit(x[tr], y_all[tr]); val.append(f1_score(y_all[va], m.predict(x[va]), average='macro'))
    C = [0.01, 0.1, 1.0][int(np.argmax(val))]
    rows = []
    for name, mask, v1 in [('clear_baseline', tr, (0.6070, 0.0539)), ('winter_adapted', tr | wi, (0.6873, 0.1752))]:
        p = logistic_head(C).fit(x[mask], y_all[mask]).predict_proba(x[te])[:, 1]; m = metrics(y_all[te], p)
        rows.append(dict(model=name, v1_macro_f1=v1[0], v2_macro_f1=round(m['macro_f1'], 4),
                         v1_false_free=v1[1], v2_false_free=round(m['false_free_rate'], 4), test_crops=int(te.sum())))
    return pd.DataFrame(rows)

replication = pilot_replication(); display(replication)
replication.to_csv(RES / 'table_v1_replication.csv', index=False)

## 10. Experimental protocol: leave-one-snow-camera-out with an independent winter validation camera
For every ordered pair (test camera $t$, validation camera $v$), with $t\ne v$ drawn from {cam1, …, cam4} (12 folds):
* **test** = snow frames of $t$. **validation** = snow frames of $v$.
* **training pool** = all frames (snow and non-snow) of every view except $t$ and $v$. No frame from the test camera, including its clear-weather frames, is ever used for training or selection.

| ID | Model | Training data | Selection of $C$ and $\tau$ |
|---|---|---|---|
| B0 | always "occupied" | — | — |
| B1 | clear-only, source-selected (v1-style) | non-snow frames | $C$ by grouped 3-fold CV on clear frames; $\tau=0.5$ |
| B2 | clear-only, winter-calibrated | non-snow frames | $C$, $\tau$ on validation camera $v$ |
| S1 | synthetic snow augmentation | non-snow + synthetic-snow copies | on $v$; mean over 3 augmentation seeds |
| W1 | **supervised winter adaptation** | non-snow + real snow frames | on $v$ |
| W2 | size-matched winter adaptation | same number of crops as B2 (½ clear, ½ winter) | on $v$; 5 subsample seeds |

$C$ is chosen by AP(free) on the validation data, and $\tau$ by the two rules in §2 (`f1` and `safe`). The headline numbers pool the predictions from all 12 folds. 95 % confidence intervals come from a **frame-cluster bootstrap stratified by test camera**: whole frames are resampled, so the CIs respect the fact that all spaces in one frame are dependent. Bootstrap replicates are paired across models.

In [ ]:
def fit_select(train_idx, val_idx, Xmat, Xval=None):
    """Fit one head per C, choose C by AP(free) on validation; return the fitted model and thresholds."""
    Xval = Xmat if Xval is None else Xval
    best = None
    for C in C_GRID:
        m = logistic_head(C).fit(Xmat[train_idx], y_all[train_idx])
        pv = m.predict_proba(Xval[val_idx])[:, 1]
        ap = average_precision_score(1 - y_all[val_idx], 1 - pv) if 0 < y_all[val_idx].sum() < len(val_idx) else -np.inf
        if best is None or ap > best[0]:
            best = (ap, C, m, pv)
    _, C, m, pv = best
    return m, C, choose_thresholds(y_all[val_idx], pv)

def fit_stack(Xparts, yparts, val_X, val_y, C_list=C_GRID):
    best = None
    Xt, yt = np.concatenate(Xparts), np.concatenate(yparts)
    for C in C_list:
        m = logistic_head(C).fit(Xt, yt); pv = m.predict_proba(val_X)[:, 1]
        ap = average_precision_score(1 - val_y, 1 - pv) if 0 < val_y.sum() < len(val_y) else -np.inf
        if best is None or ap > best[0]:
            best = (ap, C, m, pv)
    _, C, m, pv = best
    return m, C, choose_thresholds(val_y, pv)

SNOW_VIEWS = [v for v in ['cam1', 'cam2', 'cam3', 'cam4'] if ((crops.view == v) & crops.snow).any()]
FOLDS = [(t, v) for t in SNOW_VIEWS for v in SNOW_VIEWS if t != v]
clear_pos = {g: i for i, g in enumerate(clear_idx_all)}       # crop index -> row in X_AUG

def run_fold(t: str, v: str, feat: str = 'bbox', models=('B0', 'B1', 'B2', 'S1', 'W1', 'W2'), lc=True) -> list:
    x = X[feat]
    test = np.where((crops.view == t) & crops.snow)[0]; val = np.where((crops.view == v) & crops.snow)[0]
    pool = ~crops.view.isin([t, v])
    clear = np.where(pool & ~crops.snow)[0]; winter = np.where(pool & crops.snow)[0]
    out = []
    def record(model, seed, p, taus, C, n_train):
        for policy, tau in taus.items():
            out.append(dict(test_view=t, val_view=v, feature=feat, model=model, seed=seed, policy=policy, tau=tau, C=C,
                            n_train=n_train, idx=test, p=p))
    if 'B0' in models:
        record('B0', 0, np.ones(len(test)), {'default': 0.5}, None, 0)
    if len(clear) == 0:
        return out
    if 'B1' in models:
        groups = crops.frame.to_numpy()[clear]; scores = []
        for C in C_GRID:
            s = []
            for a, b in GroupKFold(n_splits=min(3, len(set(groups)))).split(clear, groups=groups):
                m = logistic_head(C).fit(x[clear[a]], y_all[clear[a]])
                s.append(average_precision_score(1 - y_all[clear[b]], 1 - m.predict_proba(x[clear[b]])[:, 1])
                         if 0 < y_all[clear[b]].sum() < len(b) else 0)
            scores.append(np.mean(s))
        C = C_GRID[int(np.argmax(scores))]
        p = logistic_head(C).fit(x[clear], y_all[clear]).predict_proba(x[test])[:, 1]
        record('B1', 0, p, {'default': 0.5}, C, len(clear))
    if 'B2' in models:
        m, C, taus = fit_select(clear, val, x)
        record('B2', 0, m.predict_proba(x[test])[:, 1], taus, C, len(clear))
    if 'S1' in models and feat == 'bbox':
        rows = [clear_pos[i] for i in clear]
        for s in AUG_SEEDS:
            m, C, taus = fit_stack([x[clear], X_AUG[s][rows]], [y_all[clear], y_all[clear]], x[val], y_all[val])
            record('S1', s, m.predict_proba(x[test])[:, 1], taus, C, 2 * len(clear))
    if 'W1' in models and len(winter):
        m, C_w1, taus = fit_select(np.concatenate([clear, winter]), val, x)
        record('W1', 0, m.predict_proba(x[test])[:, 1], taus, C_w1, len(clear) + len(winter))
        if lc and feat == 'bbox':  # learning curve: fraction of winter *frames*, C fixed to W1's choice
            wframes = np.array(sorted(set(crops.frame.to_numpy()[winter])))
            for frac in LC_FRACTIONS:
                for s in SUBSAMPLE_SEEDS:
                    pick = set(np.random.default_rng([s, 7]).choice(wframes, max(1, int(round(frac * len(wframes)))), replace=False))
                    w = winter[np.isin(crops.frame.to_numpy()[winter], list(pick))]
                    m2 =logistic_head(C_w1).fit(x[np.concatenate([clear, w])], y_all[np.concatenate([clear, w])])
                    taus2 = choose_thresholds(y_all[val], m2.predict_proba(x[val])[:, 1])
                    record(f'LC{frac:.2f}', s, m2.predict_proba(x[test])[:, 1], taus2, C_w1, len(clear) + len(w))
    if 'W2' in models and len(winter) and feat == 'bbox':
        n = len(clear)
        for s in SUBSAMPLE_SEEDS:
            rng = np.random.default_rng([s, 11])
            def take(idx, k):
                fr = rng.permutation(np.array(sorted(set(crops.frame.to_numpy()[idx])))); chosen, cnt = [], 0
                for f in fr:
                    if cnt >= k: break
                    sel = idx[crops.frame.to_numpy()[idx] == f]; chosen.append(sel); cnt += len(sel)
                return np.concatenate(chosen) if chosen else idx[:0]
            tr = np.concatenate([take(clear, n // 2), take(winter, n - n // 2)])
            m, C, taus = fit_select(tr, val, x)
            record('W2', s, m.predict_proba(x[test])[:, 1], taus, C, len(tr))
    return out

t0 = time.time(); RUNS = []
for k, (t, v) in enumerate(FOLDS):
    RUNS += run_fold(t, v)
    print(f'fold {k + 1}/{len(FOLDS)} test={t} val={v} done ({time.time() - t0:.0f}s)', flush=True)
TIMING['main_experiment_s'] = time.time() - t0

def long_predictions(runs) -> pd.DataFrame:
    parts = []
    for r in runs:
        parts.append(pd.DataFrame(dict(test_view=r['test_view'], val_view=r['val_view'], feature=r['feature'], model=r['model'],
                                       seed=r['seed'], policy=r['policy'], tau=r['tau'], C=r['C'], n_train=r['n_train'],
                                       crop=r['idx'], frame=crops.frame.to_numpy()[r['idx']], spot=crops.spot.to_numpy()[r['idx']],
                                       illum=crops.illum.to_numpy()[r['idx']], label=y_all[r['idx']], p=r['p'])))
    return pd.concat(parts, ignore_index=True)

PRED = long_predictions(RUNS)
PRED.to_csv(RES / 'predictions_all_folds.csv.gz', index=False)
print(f'{len(PRED):,} prediction rows saved; time {TIMING["main_experiment_s"]:.0f}s')

## 11. Results: pooled comparison with frame-cluster bootstrap CIs

In [ ]:
KEY_METRICS = ['ap_free', 'macro_f1', 'balanced_acc', 'false_free_rate', 'free_recall', 'free_precision']

def select(pred, model, policy, feature='bbox', **filters):
    d = pred[(pred.model == model) & (pred.policy == policy) & (pred.feature == feature)]
    for k, val in filters.items():
        d = d[d[k] == val]
    return d

def seed_mean(d: pd.DataFrame) -> pd.DataFrame:
    """Average probabilities and decisions over seeds -> one row per (fold, crop); decision = majority over seeds."""
    if d.seed.nunique() <= 1:
        return d.assign(dec=(d.p >= d.tau).astype(float))
    d = d.assign(dec=(d.p >= d.tau).astype(float))
    return d.groupby(['test_view', 'val_view', 'crop', 'frame', 'illum', 'label'], as_index=False).agg(p=('p', 'mean'), dec=('dec', 'mean'))

def metrics_from(d: pd.DataFrame) -> dict:
    return fast_metrics(d.label.to_numpy(), d.p.to_numpy(), d.dec.to_numpy())

def paired_bootstrap(tables: dict, n_boot: int = N_BOOT, seed: int = SEED) -> dict:
    """tables: name -> seed-averaged predictions on identical (fold, crop) rows.
    Whole frames are resampled with replacement within each test camera; the same resample is used for every model."""
    rng = np.random.default_rng(seed)
    keyed = {n: t.sort_values(['test_view', 'val_view', 'crop']).reset_index(drop=True) for n, t in tables.items()}
    ref = next(iter(keyed.values()))
    for n, t in keyed.items():
        assert len(t) == len(ref) and (t.crop.to_numpy() == ref.crop.to_numpy()).all(), f'{n} is not aligned'
    arrays = {n: (t.label.to_numpy(), t.p.to_numpy(), t.dec.to_numpy()) for n, t in keyed.items()}
    by_cam = defaultdict(list)
    for (cam, _), rows in ref.groupby(['test_view', 'frame']).indices.items():
        by_cam[cam].append(rows)
    boots = {n: defaultdict(list) for n in tables}
    for _ in range(n_boot):
        idx = np.concatenate([np.concatenate([lst[i] for i in rng.integers(0, len(lst), len(lst))]) for lst in by_cam.values()])
        for n, (y, p, dec) in arrays.items():
            m = fast_metrics(y[idx], p[idx], dec[idx])
            for k in KEY_METRICS:
                boots[n][k].append(m[k])
    return {n: {k: np.array(v, dtype=float) for k, v in b.items()} for n, b in boots.items()}

def ci(a):
    a = a[~np.isnan(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)

COMPARE = [('B0', 'default'), ('B1', 'default'), ('B2', 'f1'), ('B2', 'safe'), ('S1', 'f1'), ('S1', 'safe'),
           ('W1', 'f1'), ('W1', 'safe'), ('W2', 'f1'), ('W2', 'safe')]
TABLES = {f'{m}/{p}': seed_mean(select(PRED, m, p)) for m, p in COMPARE if len(select(PRED, m, p))}
BOOT = paired_bootstrap(TABLES)
rows = []
for name, t in TABLES.items():
    m = metrics_from(t); row = {'model/policy': name, 'n_test_rows': len(t)}
    for k in KEY_METRICS:
        lo, hi = ci(BOOT[name][k]); row[k] = f'{m[k]:.3f} [{lo:.3f}, {hi:.3f}]'; row[k + '_value'] = m[k]
    rows.append(row)
MAIN = pd.DataFrame(rows)
display(MAIN[['model/policy'] + KEY_METRICS])
MAIN.to_csv(RES / 'table_main_results.csv', index=False)

def delta(a, b, k):
    d = BOOT[a][k] - BOOT[b][k]; d = d[~np.isnan(d)]
    pa, pb = metrics_from(TABLES[a])[k], metrics_from(TABLES[b])[k]
    return pa - pb, (np.percentile(d, 2.5), np.percentile(d, 97.5)) if len(d) else (np.nan, np.nan)

DELTAS = []
for a, b in [('W1/safe', 'B2/safe'), ('W1/safe', 'S1/safe'), ('W1/safe', 'W2/safe'), ('S1/safe', 'B2/safe'), ('B2/f1', 'B1/default'), ('W1/f1', 'B2/f1')]:
    if a in BOOT and b in BOOT:
        for k in ['ap_free', 'free_recall', 'false_free_rate', 'macro_f1']:
            d, (lo, hi) = delta(a, b, k); DELTAS.append(dict(comparison=f'{a} − {b}', metric=k, delta=round(d, 4), ci_low=round(lo, 4), ci_high=round(hi, 4)))
DELTAS = pd.DataFrame(DELTAS); display(DELTAS); DELTAS.to_csv(RES / 'table_paired_deltas.csv', index=False)

## 12. Per-camera and per-illumination results (H3) and operating curves

In [ ]:
def per_group(models, group: str) -> pd.DataFrame:
    out = []
    for name in models:
        t = TABLES[name]
        for g, d in t.groupby(group):
            m = metrics_from(d); out.append({'model/policy': name, group: g, 'frames': d.frame.nunique(), 'n_free': m['n_free'],
                                             **{k: round(m[k], 3) for k in KEY_METRICS}})
    return pd.DataFrame(out)

PER_CAM = per_group(['B1/default', 'B2/safe', 'S1/safe', 'W1/safe'], 'test_view')
PER_CAM['small_free_class'] = PER_CAM.n_free < MIN_FREE_FOR_CAMERA_F1 * 3   # 3 validation choices per test camera
display(PER_CAM); PER_CAM.to_csv(RES / 'table_per_camera.csv', index=False)
PER_ILLUM = per_group(['B1/default', 'B2/safe', 'S1/safe', 'W1/safe'], 'illum')
display(PER_ILLUM); PER_ILLUM.to_csv(RES / 'table_per_illumination.csv', index=False)

# H3: AP(free) gain W1 vs B2 inside each illumination stratum, with stratum-restricted bootstrap
H3 = []
for g in sorted(TABLES['W1/safe'].illum.unique()):
    sub = {n: TABLES[n][TABLES[n].illum == g] for n in ['W1/safe', 'B2/safe']}
    if sub['W1/safe'].label.nunique() < 2:
        continue
    b = paired_bootstrap(sub, n_boot=max(200, N_BOOT // 4))
    d = b['W1/safe']['ap_free'] - b['B2/safe']['ap_free']
    H3.append(dict(illum=g, frames=sub['W1/safe'].frame.nunique(),
                   delta_ap_free=metrics_from(sub['W1/safe'])['ap_free'] - metrics_from(sub['B2/safe'])['ap_free'],
                   ci_low=ci(d)[0], ci_high=ci(d)[1]))
H3 = pd.DataFrame(H3); display(H3); H3.to_csv(RES / 'table_h3_illumination.csv', index=False)

# Operating curves: free recall vs false-free rate (pooled, threshold-free view)
fig, ax = plt.subplots(figsize=(5.2, 4))
for name, style in [('B2/safe', '-'), ('S1/safe', '--'), ('W1/safe', '-')]:
    t = TABLES[name]; ths = np.quantile(t.p, np.linspace(0, 1, 300))
    pts = [metrics(t.label.to_numpy(), t.p.to_numpy(), th, with_ap=False) for th in ths]
    ax.plot([q['false_free_rate'] for q in pts], [q['free_recall'] for q in pts], style, label=name.split('/')[0])
    m = metrics_from(t); ax.plot(m['false_free_rate'], m['free_recall'], 'o', color=ax.lines[-1].get_color())
ax.axvline(SAFE_FFR, color='grey', lw=0.8, ls=':'); ax.set_xlim(0, 0.4)
ax.set_xlabel('false-free rate (occupied reported free)'); ax.set_ylabel('free-space recall'); ax.legend(frameon=False)
ax.set_title('Pooled operating curves (single global τ).\nDots: validation-selected τ_safe (fold-specific, so they need not lie on the curve)', fontsize=8)
plt.tight_layout(); plt.savefig(FIG / 'fig2_operating_curves.png', dpi=200); plt.savefig(FIG / 'fig2_operating_curves.pdf'); plt.show()

## 13. Ablations: amount of winter data, training-set size, crop masking

In [ ]:
lc_rows = []
for name in ['B2'] + [f'LC{f:.2f}' for f in LC_FRACTIONS] + ['W1']:
    d = select(PRED, name, 'safe')
    if d.empty:
        continue
    frac = 0.0 if name == 'B2' else (1.0 if name == 'W1' else float(name[2:]))
    for s, ds in d.groupby('seed'):
        m = metrics_from(ds.assign(dec=(ds.p >= ds.tau).astype(float)))
        lc_rows.append(dict(winter_fraction=frac, seed=s, **{k: m[k] for k in KEY_METRICS}))
LC = pd.DataFrame(lc_rows); LC.to_csv(RES / 'table_learning_curve.csv', index=False)
lc_summary = LC.groupby('winter_fraction')[['ap_free', 'free_recall', 'false_free_rate']].agg(['mean', 'std']).round(3)
display(lc_summary)
fig, ax = plt.subplots(figsize=(5.2, 3.6))
g = LC.groupby('winter_fraction').ap_free
ax.errorbar(g.mean().index, g.mean(), yerr=g.std().fillna(0), marker='o', capsize=3)
ax.set_xlabel('fraction of winter training frames used'); ax.set_ylabel('AP (free class), pooled test')
ax.set_title('Learning curve (mean ± s.d. over subsample seeds)', fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'fig3_learning_curve.png', dpi=200); plt.savefig(FIG / 'fig3_learning_curve.pdf'); plt.show()

if RUN_MASK_ABLATION:
    MASK_RUNS = []
    for t, v in FOLDS:
        MASK_RUNS += run_fold(t, v, feat='mask', models=('B2', 'W1'), lc=False)
    PRED_MASK = long_predictions(MASK_RUNS)
    abl = []
    for feat, pred in [('bbox', PRED), ('mask', PRED_MASK)]:
        for mname in ['B2', 'W1']:
            m = metrics_from(seed_mean(select(pred, mname, 'safe', feature=feat)))
            abl.append(dict(crop=feat, model=mname, **{k: round(m[k], 3) for k in KEY_METRICS}))
    ABL = pd.DataFrame(abl); display(ABL); ABL.to_csv(RES / 'table_ablation_mask.csv', index=False)

c_choice = PRED[(PRED.policy == 'safe')].drop_duplicates(['test_view', 'val_view', 'model', 'seed']).groupby('model').C.value_counts()
print('Selected C per model (count over folds/seeds):'); display(c_choice.unstack(fill_value=0))

## 14. Error analysis (W1, safe operating point)

In [ ]:
w1 = select(PRED, 'W1', 'safe').copy(); w1['pred'] = (w1.p >= w1.tau).astype(int)
w1['error'] = np.select([(w1.label == 1) & (w1.pred == 0), (w1.label == 0) & (w1.pred == 1)], ['false_free', 'false_occupied'], 'correct')
err_by = w1.groupby(['test_view', 'illum']).error.value_counts(normalize=True).unstack(fill_value=0).round(3)
display(err_by); err_by.to_csv(RES / 'table_errors_by_view_illum.csv')
spot_err = (w1.groupby(['test_view', 'spot']).apply(lambda d: pd.Series({'n': len(d), 'false_free': (d.error == 'false_free').mean(),
                                                                        'occupied_share': d.label.mean(), 'box_area': crops.box_area.iloc[d.crop.iloc[0]]}),
                                                     include_groups=False).reset_index())
print('Spots with the highest false-free rate (spatial concentration of errors):')
display(spot_err.sort_values('false_free', ascending=False).head(10))
print('Spearman correlation between crop area and false-free rate:', round(spot_err[['box_area', 'false_free']].corr('spearman').iloc[0, 1], 3))
spot_err.to_csv(RES / 'table_errors_by_spot.csv', index=False)

worst = w1[w1.error == 'false_free'].assign(conf=lambda d: d.tau - d.p).sort_values('conf', ascending=False).drop_duplicates('crop').head(12)
if len(worst):
    fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
    for ax, (_, r) in zip(axes.flat, worst.iterrows()):
        with Image.open(BASE / crops.bbox.iloc[r.crop]) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'{r.test_view} {r.illum}\np={r.p:.2f} τ={r.tau:.2f}', fontsize=7); ax.axis('off')
    for ax in axes.flat[len(worst):]: ax.axis('off')
    plt.suptitle('Most confident false-free errors (truly occupied, predicted free)', fontsize=10)
    plt.tight_layout(); plt.savefig(FIG / 'fig4_false_free_examples.png', dpi=200); plt.show()

## 15. Hypothesis tests (generated from the numbers above, not hand-written)

In [ ]:
def verdict(cond):
    return 'SUPPORTED' if cond is True else ('NOT SUPPORTED' if cond is False else 'INCONCLUSIVE')
lines, H = [], {}
if {'W1/safe', 'B2/safe'} <= set(BOOT):
    d, (lo, hi) = delta('W1/safe', 'B2/safe', 'ap_free')
    H['H1'] = lo > 0 if not np.isnan(lo) else None
    lines.append(f"**H1** ΔAP(free) W1−B2 = {d:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}] → {verdict(H['H1'])}")
    w1m = metrics_from(TABLES['W1/safe']); dr, (rlo, rhi) = delta('W1/safe', 'B2/safe', 'free_recall')
    ok_ffr = w1m['false_free_rate'] <= SAFE_FFR + H2_TOLERANCE
    H['H2'] = bool(ok_ffr and rlo > 0)
    lines.append(f"**H2** W1 test FFR = {w1m['false_free_rate']:.3f} (budget {SAFE_FFR + H2_TOLERANCE:.2f}); Δ free recall W1−B2 = {dr:+.3f} [{rlo:+.3f}, {rhi:+.3f}] → {verdict(H['H2'])}")
if len(H3):
    gains = H3.set_index('illum')
    strata = [s for s in ['day', 'night'] if s in gains.index]
    H['H3'] = all(gains.loc[s, 'ci_low'] > 0 for s in strata) if len(strata) == 2 else None
    lines.append('**H3** ' + '; '.join(f"{s}: ΔAP={gains.loc[s, 'delta_ap_free']:+.3f} [{gains.loc[s, 'ci_low']:+.3f}, {gains.loc[s, 'ci_high']:+.3f}]" for s in gains.index) + f" → {verdict(H['H3'])}")
if {'W1/safe', 'S1/safe'} <= set(BOOT):
    d, (lo, hi) = delta('W1/safe', 'S1/safe', 'ap_free'); H['H4'] = lo > 0 if not np.isnan(lo) else None
    lines.append(f"**H4** ΔAP(free) W1−S1 = {d:+.3f} [{lo:+.3f}, {hi:+.3f}] → {verdict(H['H4'])}")
if SMALL_FREE_VIEWS:
    lines.append(f"Note: free class too small for stable per-camera F1 on {', '.join(SMALL_FREE_VIEWS)}; pooled metrics are primary.")
display(Markdown('\n\n'.join(lines)))
_ = (RES / 'hypotheses.md').write_text('\n\n'.join(lines))

## 16. ✋ Heavy-snowfall / hidden-marking subset (manual, two independent annotators)
The `Snow` tag cannot tell falling snow from lying snow, and it does not say whether markings exist under the snow. **Each author labels the same frames independently, without seeing any predictions.** Labelling all 440 snow frames takes about 1–1.5 h per person.
* `snowfall`: `none_visible` / `light` / `heavy`. Heavy means visible falling snow with clearly reduced visibility. White ground alone is not heavy snowfall.
* `markings`: `visible` / `partial` / `hidden` / `unmarked` / `unknown`. Hidden means lines are known to exist, for example from a clear frame of the same camera (shown on the right), but are not visible.

Labels are saved to `results/condition_review.csv`. The next cell reports Cohen's κ and evaluates the consensus subset, i.e. frames that both annotators labelled heavy **and** hidden.

In [ ]:
def condition_review():
    import ipywidgets as W
    path = RES / 'condition_review.csv'
    snow_frames = sorted(frames[frames.snow].frame)
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=['frame', 'reviewer', 'snowfall', 'markings', 'evidence'])
    choose = W.Dropdown(options=snow_frames, description='Frame:', layout=W.Layout(width='420px'))
    reviewer = W.Dropdown(options=['A1', 'A2'], description='Reviewer:')
    snowfall = W.Dropdown(options=['unknown', 'none_visible', 'light', 'heavy'], description='Snowfall:')
    markings = W.Dropdown(options=['unknown', 'visible', 'partial', 'hidden', 'unmarked'], description='Markings:')
    note = W.Text(description='Evidence:', layout=W.Layout(width='420px')); btn = W.Button(description='Save + next', button_style='success')
    out = W.Output()
    def show(*_):
        with out:
            out.clear_output(wait=True)
            view = frames.set_index('frame').view[choose.value]
            ref = frames[(frames.view == view) & ~frames.snow & (frames.illum == 'day')]
            pics = [DATA / 'frames' / choose.value] + ([DATA / 'frames' / ref.frame.iloc[0]] if len(ref) else [])
            fig, axes = plt.subplots(1, len(pics), figsize=(7 * len(pics), 4.5))
            for ax, pth in zip(np.atleast_1d(axes), pics):
                with Image.open(pth) as im: ax.imshow(im.convert('RGB'))
                ax.set_title(pth.name + (' (clear reference)' if pth != pics[0] else ''), fontsize=9); ax.axis('off')
            plt.show()
    def save(_):
        nonlocal done
        done = done[~((done.frame == choose.value) & (done.reviewer == reviewer.value))]
        done = pd.concat([done, pd.DataFrame([dict(frame=choose.value, reviewer=reviewer.value, snowfall=snowfall.value,
                                                   markings=markings.value, evidence=note.value)])], ignore_index=True)
        done.to_csv(path, index=False); i = snow_frames.index(choose.value)
        choose.value = snow_frames[min(i + 1, len(snow_frames) - 1)]
    choose.observe(show, names='value'); btn.on_click(save)
    display(W.VBox([W.HTML('<b>Label the raw frame only. Predictions are hidden on purpose.</b>'), reviewer, choose, snowfall, markings, note, btn, out])); show()

if not SMOKE_TEST:
    condition_review()

In [ ]:
def evaluate_condition_subset():
    path = RES / 'condition_review.csv'
    if not path.exists():
        print('✋ No condition labels yet; heavy/hidden results are PENDING and no numbers are reported.'); return None
    lab = pd.read_csv(path); wide = lab.pivot_table(index='frame', columns='reviewer', values=['snowfall', 'markings'], aggfunc='first')
    if not {('snowfall', 'A1'), ('snowfall', 'A2')} <= set(wide.columns):
        print('✋ Need labels from both reviewers (A1 and A2).'); return None
    both = wide.dropna()
    for field in ['snowfall', 'markings']:
        print(f'{field}: {len(both)} frames labelled by both; Cohen κ = {cohen_kappa_score(both[(field, "A1")], both[(field, "A2")]):.3f}')
    heavy_hidden = both[(both[('snowfall', 'A1')] == 'heavy') & (both[('snowfall', 'A2')] == 'heavy') &
                        (both[('markings', 'A1')] == 'hidden') & (both[('markings', 'A2')] == 'hidden')].index
    print(f'Consensus heavy + hidden frames: {len(heavy_hidden)}')
    if len(heavy_hidden) < 10:
        print('Fewer than 10 consensus frames: report counts only, no subset metrics.'); return None
    rows = []
    for name in ['B1/default', 'B2/safe', 'S1/safe', 'W1/safe']:
        d = TABLES[name][TABLES[name].frame.isin(heavy_hidden)]
        if d.label.nunique() == 2:
            m = metrics_from(d); rows.append({'model/policy': name, 'frames': d.frame.nunique(), **{k: round(m[k], 3) for k in KEY_METRICS}})
    res = pd.DataFrame(rows); display(res); res.to_csv(RES / 'table_heavy_hidden_subset.csv', index=False); return res

HEAVY_HIDDEN = evaluate_condition_subset()

## 17. Computational cost

In [ ]:
n_params = sum(p.numel() for p in encoder().parameters())
COST = dict(encoder='ResNet-18 (frozen, ImageNet)', encoder_params_M=round(n_params / 1e6, 2), head_params=513,
            embed_ms_per_crop=round(TIMING['embed_ms_per_crop'], 2), crops=len(crops), device=VERSIONS['device'],
            main_experiment_s=round(TIMING['main_experiment_s'], 1))
display(pd.Series(COST)); _ = (RES / 'cost.json').write_text(json.dumps(COST, indent=1))

## 18. Discussion, limitations and conclusions (to be finalised with the v2 numbers)
Read §11–§16 before writing any claim. The paper must state the following limitations whatever the results:
* **Geometry prior.** The slot polygons are given. The method does not locate hidden lines; it makes them unnecessary for *fixed* cameras that are annotated once.
* **Four snow cameras from one dataset.** Leave-one-camera-out uses each camera as a test once, but the four cameras may share a site, a season or a sensor. External validity, for example on Astana parking lots, is untested.
* **Frozen encoder.** Only the linear head adapts. Fine-tuning the backbone or using a modern detector could change the ranking of the approaches.
* **Dependence.** Consecutive frames and parked vehicles repeat. The bootstrap resamples frames, but frames close in time are still correlated, so the CIs are optimistic.
* **Synthetic snow is heuristic.** S1 represents one augmentation design, not every possible one.
* **Condition labels are visual judgements.** Heavy snowfall is judged by eye; no precipitation measurements exist.

## 19. Reproducibility: provenance and results archive

In [ ]:
def sha(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()
PROVENANCE = dict(date=time.strftime('%Y-%m-%d %H:%M'), versions=VERSIONS, seed=SEED, smoke_test=SMOKE_TEST,
                  config=dict(CROP_SIZE=CROP_SIZE, C_GRID=C_GRID, SAFE_FFR=SAFE_FFR, AUG_SEEDS=AUG_SEEDS,
                              SUBSAMPLE_SEEDS=SUBSAMPLE_SEEDS, LC_FRACTIONS=LC_FRACTIONS, N_BOOT=N_BOOT),
                  dataset=dict(name='SPKLv2', source='https://github.com/Eighonet/parking-research', gdrive_id=GDRIVE_FILE_ID,
                               frames=len(frames), crops=len(crops), missing_tagged_frames=len(missing)),
                  folds=FOLDS, files={p.name: sha(p) for p in sorted(RES.glob('*.csv')) + [RES / 'crops_manifest.csv']})
_ = (RES / 'provenance.json').write_text(json.dumps(PROVENANCE, indent=1, default=str))
archive = shutil.make_archive(str(BASE / 'SnowParking_v2_results'), 'zip', BASE, 'results')
print('Results archive:', archive)
try:
    from google.colab import files  # noqa
    files.download(archive)
except Exception:
    pass